**Imports**

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import Bounds # Bounds for Optimization
from scipy.optimize import LinearConstraint # Linear Constraints for Optimization
from scipy.optimize import milp # MILP Optimization Method
import tkinter as tk
from tkinter import *

**Data Formating/Input Reading**

In [2]:
input_file_1 = "Final/student_data_1.csv"
input_file_2 = "Final/project_data.csv"

student_data = pd.read_csv(input_file_1)
project_data = pd.read_csv(input_file_2)

project_names = project_data["project_name"]
names = student_data["name"]

# Capstone selections for individual students.
pick1 = student_data["pick_1"]
pick2 = student_data["pick_2"]
pick3 = student_data["pick_3"]
pick4 = student_data["pick_4"]

**Optimization Method (MILP)**

In [3]:
# Get student names, project names, and student rankings for optimization. 
num_students = len(names)
num_projects = len(project_names)
rankings = student_data[["pick_1", "pick_2", "pick_3", "pick_4"]].to_numpy()

# Higher preference rankings get higher values (1 being highest value).
value = np.zeros((num_students, num_projects))

for i in range(num_students):
    for rank_pos in range(rankings.shape[1]):  # 4 positions/picks
        project_index = int(rankings[i, rank_pos]) - 1
        value[i, project_index] = num_projects - rank_pos

# Maximum amounts of students in each capstone group 
# (equal number of students in groups and remainder of students spread out to first few available projects).
base = num_students // num_projects
remainder = num_students % num_projects

capacities = []
for i in range(num_projects):
    if i < remainder: capacities.append(base + 1)
    else: capacities.append(base)

capacities = np.array(capacities)

# Objective Function: Negate values for maximizing student preferences per each project.
c = -value.flatten()

# List to store constraints. 
constraints = []

# Equality Constraint Matrix
A_eq_students = np.zeros((num_students, num_students * num_projects))

# Each row is a student and each student is constrained to be a part of one project.
for i in range(num_students):
    for j in range(num_projects):
        A_eq_students[i, i * num_projects + j] = 1

# Each student assigned to one project.
b_eq_students = np.ones(num_students)

# Add equality constraints to constraint list.
constraints.append(LinearConstraint(A_eq_students, b_eq_students, b_eq_students))

# Inequality Constraint Matrix
A_ub_projects = np.zeros((num_projects, num_students * num_projects))

# Each column is a project and counts how many students are attached to it.
for j in range(num_projects):
    for i in range(num_students):
        A_ub_projects[j, i * num_projects + j] = 1

# Add inequality constraints to constraint list.
constraints.append(LinearConstraint(A_ub_projects, -np.inf, capacities))

# Each variable is binary: 0 not in group, 1 is + integrality is binary
bounds = Bounds(0, 1)
integrality = np.ones(num_students * num_projects, dtype=int)

# Run the MILP optimization method. Reshape the solution into a matrix.
result = milp(c=c, constraints=constraints, integrality=integrality, bounds=bounds)
x = result.x.reshape((num_students, num_projects)).astype(int)

**Display Names as They are Chosen/Assigned**

In [4]:
def on_select(event):
    first_pick = []
    second_pick = []
    assigned = []
    selected_indices = event.widget.curselection()
    if selected_indices:
        index = selected_indices[0]     #item index
        value = event.widget.get(index) #project name
        
        for i in range(len(names)):
            if pick1[i] == index+1:
                first_pick.append(names[i])
        label1.configure(text = f"Students that chose {value} as their first pick: {str(first_pick)}") # type: ignore

        for i in range(len(names)):
            if pick2[i] == index+1:
                second_pick.append(names[i])
        label2.configure(text = f"Students that chose {value} as their second pick: {str(second_pick)}") # type: ignore

        for i in range(len(names)):
            if x[i, index] == 1:
                assigned.append(names[i])
        label3.configure(text = f"Assigned to {value}: {str(assigned)}")

**UI Creation**

In [5]:
root = tk.Tk()
root.title("App test")
height = 360
width = 720
screenheight = root.winfo_screenheight()
screenwidth = root.winfo_screenwidth()
alignscreen = '%dx%d' % (screenwidth/1.1,screenheight/1.5)
root.geometry(alignscreen)


#top bar menu stuff
menu = tk.Menu(root)
root.config(menu=menu)

settingsmenu = tk.Menu(menu)
menu.add_cascade(label="Settings", menu=settingsmenu)
settingsmenu.add_command(label="Settings")
settingsmenu.add_separator()
settingsmenu.add_command(label="Exit",command=root.quit)

helpmenu = tk.Menu(menu)
menu.add_cascade(label="Help", menu=helpmenu)
helpmenu.add_command(label="About")


#initializing UI items
label1 = tk.Label(root, text="Label1")
label2 = tk.Label(root, text="Label2")
label3 = tk.Label(root, text="Label3")
lb = tk.Listbox(root)
for i in range(len(project_data["project_name"])):
    lb.insert(i, project_names[i])

lb.bind('<<ListboxSelect>>', on_select)

lb.grid(row=0, column=0, sticky="w")
label1.grid(row=1, column=0, sticky="w")
label2.grid(row=2, column=0, sticky="w")
label3.grid(row=3, column=0, sticky="w")
root.mainloop()